In [40]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2, time, sys, os, pathlib, json

In [48]:
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_pose = mp.solutions.pose

custom_style = mp_drawing_styles.get_default_pose_landmarks_style()
custom_connections = list(mp_pose.POSE_CONNECTIONS)
excluded_landmarks = [
    mp_pose.PoseLandmark.LEFT_EYE, 
    mp_pose.PoseLandmark.RIGHT_EYE, 
    mp_pose.PoseLandmark.LEFT_EYE_INNER, 
    mp_pose.PoseLandmark.RIGHT_EYE_INNER, 
    mp_pose.PoseLandmark.LEFT_EAR,
    mp_pose.PoseLandmark.RIGHT_EAR,
    mp_pose.PoseLandmark.LEFT_EYE_OUTER,
    mp_pose.PoseLandmark.RIGHT_EYE_OUTER,
    mp_pose.PoseLandmark.NOSE,
    mp_pose.PoseLandmark.MOUTH_LEFT,
    mp_pose.PoseLandmark.MOUTH_RIGHT,
    mp_pose.PoseLandmark.RIGHT_ELBOW,
    mp_pose.PoseLandmark.RIGHT_WRIST,
    mp_pose.PoseLandmark.RIGHT_THUMB,
    mp_pose.PoseLandmark.RIGHT_PINKY,
    mp_pose.PoseLandmark.RIGHT_INDEX,
    mp_pose.PoseLandmark.LEFT_ELBOW,
    mp_pose.PoseLandmark.LEFT_WRIST,
    mp_pose.PoseLandmark.LEFT_THUMB,
    mp_pose.PoseLandmark.LEFT_PINKY,
    mp_pose.PoseLandmark.LEFT_INDEX ]
for landmark in excluded_landmarks:
    # we change the way the excluded landmarks are drawn
    custom_style[landmark] = mp_drawing_styles.DrawingSpec(color=(255,255,0), thickness=None) 
    # we remove all connections which contain these landmarks
    custom_connections = [connection_tuple for connection_tuple in custom_connections if landmark.value not in connection_tuple]

pose =  mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

video_path = "ATH_videos/ATH02/s1/PreCut01.mp4"
output_path = "test.mp4"

cv2.startWindowThread()
cap = cv2.VideoCapture(video_path)

avg_fps = []

width, height, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"avc1"), float(fps), (width, height))

landmark_summary = "landmarks_summary/" + pathlib.Path(output_path).stem + "_results.json"
os.makedirs("landmarks_summary", exist_ok=True)
frames_landmarks = []
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (width, height))

    start_time = time.time()
    rbg_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rbg_frame)
    process_time = time.time() - start_time

    mp_drawing.draw_landmarks(frame, results.pose_landmarks, connections=custom_connections, landmark_drawing_spec=custom_style)

    # Collect landmarks (x, y, z, visibility) for this frame in a serializable form
    landmarks_data = []
    # Try common attributes that store landmarks
    pose_landmarks = getattr(results, 'pose_world_landmarks', None)
    if pose_landmarks is None:
        pose_landmarks = getattr(results, 'pose_world_landmarks', None)
    # Some result variants may store landmarks under different attributes
    if pose_landmarks is None and hasattr(results, 'pose_results'):
        pose_landmarks = getattr(results.pose_results, 'world_landmark', None)

    if pose_landmarks is not None:
        landmark_list = getattr(pose_landmarks, 'landmark', None) or pose_landmarks
        if landmark_list:
            for idx, lm in enumerate(landmark_list):
                try:
                    name = mp_pose.PoseLandmark(idx).name
                    if mp_pose.PoseLandmark(idx) in excluded_landmarks:
                        continue
                except Exception:
                    name = str(idx)
                visibility = getattr(lm, 'visibility', None)
                landmarks_data.append({
                    'index': idx,
                    'name': name,
                    'x': float(lm.x),
                    'y': float(lm.y),
                    'z': float(lm.z),
                    'visibility': float(visibility) if visibility is not None else None
                })

    frames_landmarks.append({'frame': frame_idx, 'landmarks': landmarks_data})
    frame_idx += 1

    fps = 1 / process_time if process_time > 0 else 0
    avg_fps.append(fps)
    fps = sum(avg_fps) / len(avg_fps)

    video_writer.write(frame)
    cv2.imshow("Pose Estimation", frame)
    
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    break

# Write collected landmarks to JSON
summary = {
    'video': os.path.basename(video_path),
    'frame_count': frame_idx,
    'frames': frames_landmarks
}
with open(landmark_summary, 'w') as fh:
    json.dump(summary, fh, indent=2)

print(f"Wrote summary to {landmark_summary}. Processed {frame_idx} frames.")

video_writer.release()
cap.release()

cv2.destroyAllWindows()
for i in range (1,5):
    cv2.waitKey(1)

I0000 00:00:1775049156.889438   11419 gl_context.cc:369] GL version: 2.1 (2.1 INTEL-18.8.16), renderer: Intel(R) HD Graphics 6000
2026-04-01 15:12:36.978 python[1375:11419] AVF: AVAssetWriter status: Cannot Save
W0000 00:00:1775049157.363024   47370 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775049157.426449   47369 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Wrote summary to landmarks_summary/test_results.json. Processed 1 frames.


In [49]:
idx


32

In [45]:
name

'RIGHT_FOOT_INDEX'